# 04 — Model training (LightGBM cross-validation)

**Variant:** `baseline_column_transformer` · Ke et al. (2017) histogram GBDT with Pedregosa (2011) preprocessing pipeline.

> **Slurm note:** set `MAX_TRAIN_ROWS=None` and run on Matador for full-scale training per project rules.


In [ ]:
"""Shared imports for Rogii TVT pipeline notebooks."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
if not (NB_DIR / "phase_runner.py").is_file():
    NB_DIR = Path(r"/lustre/work/sweeden/agent-tracing-trace-baseline/examples/rogii/traces/preprocessing/baseline_column_transformer/notebooks")
VARIANT_DIR = NB_DIR.parent
ROGII_ROOT = Path("/lustre/work/sweeden/rogii")
sys.path.insert(0, str(NB_DIR))
if str(ROGII_ROOT) not in sys.path:
    sys.path.insert(0, str(ROGII_ROOT))

from phase_runner import (
    ARTIFACTS_ROOT,
    TRACE_INDEX,
    require_prior_phase,
    load_phase_manifest,
    trace_steps_for_phase,
    _resolve_path,
    _read_json,
    _load_train_predict,
)

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

PHASE = "04_model_training"
MAX_TRAIN_ROWS = None  # e.g. 2000 for login-node smoke test


## 1. Load prior phase configs


In [ ]:
require_prior_phase(PHASE)
p01 = load_phase_manifest("01_data_analysis")
p03 = load_phase_manifest("03_feature_engineering")
print("target:", p01["target_column"])
print("cv outputs:", list(p03["outputs"].keys()))


## 2. Target transform decision


In [ ]:
target_diag = _read_json(_resolve_path(p01["outputs"]["target_diagnosis"]))
use_log1p = bool(target_diag.get("use_log1p", False))
print("use_log1p:", use_log1p)


## 3. Estimator backend


In [ ]:
tp = _load_train_predict()
backend = "lightgbm" if tp._HAS_LGBM else "sklearn.hist_gradient_boosting"
print("backend:", backend)


## 4. Run cross-validated training


In [ ]:
from phase_runner import run_04_model_training
manifest = run_04_model_training(max_rows=MAX_TRAIN_ROWS)
print(json.dumps(manifest, indent=2))


## 5. CV RMSE by fold


In [ ]:
metrics = _read_json(_resolve_path(manifest["outputs"]["training_metrics"]))
fold_rmses = metrics.get("fold_rmses", [])
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(len(fold_rmses)), fold_rmses, marker="o")
ax.set_xlabel("fold"); ax.set_ylabel("RMSE"); ax.set_title(f"CV fold RMSE (mean={metrics.get('cv_rmse'):.3f})")
plt.tight_layout(); plt.show()


## 6. OOF prediction preview


In [ ]:
oof = np.load(_resolve_path(manifest["outputs"]["oof_predictions"]))
print("OOF shape:", oof.shape, "| finite:", np.isfinite(oof).mean())


## 7. Test prediction matrix


In [ ]:
test_mat = np.load(_resolve_path(manifest["outputs"]["test_preds_per_fold"]))
print("test_preds_per_fold:", test_mat.shape)


## 8. Transform metadata


In [ ]:
transform = _read_json(_resolve_path(manifest["outputs"]["transform"]))
print(json.dumps({k: transform[k] for k in ["use_log1p", "cv_scheme", "n_folds", "cv_rmse", "backend"]}, indent=2))


## 9. OOF vs true scatter

Visual sanity check before submission — strong public kernels always plot predicted vs actual on CV holdout.


In [ ]:
p01 = load_phase_manifest("01_data_analysis")
paths = _read_json(_resolve_path(p01["outputs"]["data_paths"]))
train_df = pd.read_csv(paths["train_csv"])
train_df = tp.align_train_target_to_schema(train_df, transform["target_column"])
y = train_df[transform["target_column"]].astype(float).values
pred = np.expm1(np.clip(oof, None, 20)) if transform.get("use_log1p") else oof
m = np.isfinite(pred) & np.isfinite(y)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y[m], pred[m], s=4, alpha=0.25)
lims = [min(y[m].min(), pred[m].min()), max(y[m].max(), pred[m].max())]
ax.plot(lims, lims, "k--", lw=0.8)
ax.set_xlabel("y_true"); ax.set_ylabel("y_pred"); ax.set_title("OOF vs true")
plt.tight_layout(); plt.show()


## 10. LightGBM hyperparameters snapshot


In [ ]:
print(json.dumps(metrics.get("params", metrics.get("lgbm_params", {})), indent=2)[:1200])


## 11. Training row count actually used


In [ ]:
print("n_train_rows:", transform.get("n_train_rows"), "max_rows:", transform.get("max_rows_applied"))


## 12. Fold RMSE distribution


In [ ]:
import pandas as pd
pd.Series(fold_rmses).describe()


## 13. Public kernel benchmark note


Top Rogii public notebooks (May 2026) report CV/public scores roughly **9.2–10.2** for strong solutions and **13–15** for starters — use `cv_rmse` to track iteration.


## 14. Compare to public LightGBM baselines


In [ ]:
print("cv_rmse:", metrics.get("cv_rmse"))


## 15. Trace steps


In [ ]:
print("trace steps:", len(trace_steps_for_phase(PHASE)))


## 16. Model artifact paths


In [ ]:
for k, v in manifest["outputs"].items():
    print(k, "→", v)


## 17. Leaderboard context (May 2026)

Top public Rogii kernels report **~9.2–9.9** public LB; starter baselines sit **13–15** CV. Track `cv_rmse` against this band while iterating features.


## 18. Handoff to evaluation

Phase 05 consumes `oof_predictions.npy` and `transform.json` for residual diagnostics.
